In [1]:
import pandas as pd

sales = pd.read_excel("../data/Online Retail.xlsx")

sales.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Data Cleaning

Before starting the analysis, the dataset was inspected for unusual values, missing values, and records that could affect the accuracy of the sales analysis.

The cleaning decisions were made based on the meaning of the data rather than automatically removing unusual or missing values.

### 1. Negative Quantities

Negative quantities were investigated because they can represent returns or cancelled transactions rather than incorrect data.

In [2]:
(sales["Quantity"] < 0).sum()

np.int64(10624)

Most negative quantities were associated with cancellation invoices. These were identified by invoice numbers beginning with `C`.

Cancellation records were kept because they represent meaningful transaction information.

In [3]:
sales["InvoiceNo"].str.startswith("C").sum()

9288

The remaining negative-quantity records were investigated to determine whether they represented meaningful sales activity.

In [4]:
sales[
    (sales["Quantity"] < 0) &
    (~sales["InvoiceNo"].str.startswith("C", na=False))
].shape

(1336, 8)

In [5]:
sales_clean = sales[
    ~(
        (sales["Quantity"] < 0) &
        (sales["UnitPrice"] == 0)
    )
].copy()

### 2. Negative Unit Prices

Negative unit prices were investigated because a product sale should not normally have a negative selling price.

In [6]:
(sales_clean["UnitPrice"] < 0).sum()

np.int64(2)

Both records were labelled `Adjust bad debt`, indicating accounting adjustments rather than actual product sales.

Therefore, these records were removed from the sales analysis.

In [7]:
sales_clean = sales_clean[
    sales_clean["UnitPrice"] >= 0
].copy()

### 3. Missing CustomerID

Missing `CustomerID` values were investigated rather than automatically removed.

These records can still be useful for overall sales, product, country, and time-based analysis. Therefore, they were retained.

For customer-level analysis, records with a known `CustomerID` will be used.

In [8]:
sales_clean["CustomerID"].isna().sum()

np.int64(133742)

### 4. Missing Product Descriptions

Missing product descriptions were investigated to determine whether these records still contained useful sales information.

In [9]:
sales_clean["Description"].isna().sum()

np.int64(592)

In [10]:
sales_clean[
    sales_clean["Description"].isna() &
    sales_clean["CustomerID"].isna() &
    (sales_clean["UnitPrice"] == 0)
].shape[0]

592

All 592 records with missing `Description` also had a missing `CustomerID` and a `UnitPrice` of 0.

Since these records provide no identifiable product, customer, or revenue information for our sales analysis, they were removed.

In [11]:
sales_clean = sales_clean[
    ~(
        sales_clean["Description"].isna() &
        sales_clean["CustomerID"].isna() &
        (sales_clean["UnitPrice"] == 0)
    )
].copy()

### 5. Duplicate Records

Exact duplicate rows were checked across all columns. We found **5,268 duplicate records**.

After investigating the duplicate rows, we confirmed that they were exact duplicates, meaning all column values were identical. Therefore, these duplicate rows were **removed** from the dataset to avoid counting the same transaction record multiple times.

```python
sales_clean = sales_clean.drop_duplicates().copy()
```

The dataset was then verified to ensure that no exact duplicate rows remained.


In [19]:
sales_clean=sales_clean.drop_duplicates().copy()

In [21]:
sales_clean.duplicated().sum()

np.int64(0)